# 📝 LangGraph 과제 LV1: 장애 인수인계와 후속 질문

**목표:** 장애 보고서에서 확인된 사실과 아직 모르는 항목을 구분하고, 운영 수칙에 근거한 인수인계 노트를 만듭니다. 이어서 담당자의 후속 질문에 앞 대화를 기억하며 답합니다.

**가상 자료:** 아래 보고서와 운영 수칙은 학습을 위해 만든 서비스 운영 사례입니다. 실제 시스템을 조회하거나 조치하지 않습니다.

<img src="images/20_assignment_lv1_graph.png" width="1050" style="background:white" alt="인수인계 그래프의 결과를 후속 대화 그래프의 State로 전달합니다. 질문 입력 뒤 답변하거나 종료하며, 답변 뒤에는 다음 입력을 받습니다.">

**만들 흐름**

- 1~6번: 보고서 -> 사실 추출 -> 컨텍스트 정리 -> 인수인계 노트
- 7~10번: 위 결과를 근거로 질문 입력 -> 답변 -> 다음 입력, `/종료`면 종료

**풀이 순서:** 1번 State → 2~4번 노드 → 5번 엣지 → 6번 실행. 이어서 7번 대화 State → 8~9번 노드 → 10번 엣지·실행으로 확장합니다.

**작성 방법:** 10문항을 순서대로 풉니다. 1·7번은 State와 초기 상태 전체를 작성하고, 나머지는 이름과 함수 골격이 있는 핵심 부분을 채웁니다. 긴 프롬프트와 구조화 출력 스키마는 제공합니다.

**모델 호출:** 6번에서 GPT 2회, 10번에서는 입력한 질문마다 1회입니다. 아래 두 질문을 사용하면 총 4회이며 `/종료`는 모델을 호출하지 않습니다. `.env`의 `OPENAI_API_KEY`와 `OPENAI_CHAT_MODEL=gpt-5.6-luna`를 준비하세요.


## 준비: 보고서·운영 수칙·모델

수치가 관찰됐다는 사실과 장애 원인이 밝혀졌다는 것은 다릅니다. 보고서에서 **확인된 사실 / 확인하지 못한 항목 / 아직 하지 않은 조치**를 먼저 구분해 읽으세요.


In [ ]:
import json
import os
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import find_dotenv, load_dotenv
from IPython.display import Image, Markdown, display
from langchain_openai import ChatOpenAI

material_dir = Path(".")
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")
load_dotenv(material_dir / ".env", override=True)
llm = ChatOpenAI(
    model=os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna"),
    use_responses_api=True,
)
print("사용 모델:", llm.model_name)

# 학습용 가상 장애 보고서와 운영 수칙입니다.
incident_report = """
사건 ID: INC-0928 / 서비스: 주문 내역 조회
09:10부터 일부 사용자가 결제 후 주문 내역을 여는 데 오래 걸린다고 신고했다.
09:10~09:25 주문 조회 요청 1200건 중 187건에서 시간 초과가 관찰됐다.
09:25 기준 응답 시간의 95백분위수는 8.2초이며 평소에는 0.8초다.
09:05에는 주문 화면 문구를 바꾸는 배포가 있었다. 배포와 지연의 인과관계는 확인하지 못했다.
장애 원인은 아직 확정하지 않았다. 결제 데이터 손실 여부는 확인 중이다.
완료된 복구 조치는 없으며 복구 예정 시각도 정하지 않았다.
""".strip()
runbook = """
[확인] 관찰 사실과 추측을 분리한다. 원인, 데이터 손실, 복구 여부는 확인 전 단정하지 않는다.
[다음 확인] 주문 조회 오류율과 응답 시간의 최근 5분 추이를 확인하고 배포 전후 지표를 비교한다.
[조치] 재시작이나 롤백은 원인과 영향을 검토한 뒤 담당자의 승인으로 진행한다.
[공유] 확인된 상황과 아직 모르는 항목을 구분해 15분 간격으로 공유한다.
""".strip()

# LLM이 반환할 사실 목록을 검증하는 출력 형식
from pydantic import BaseModel, Field

class IncidentFacts(BaseModel):
    symptoms: list[str]
    affected_scope: str
    confirmed_cause: str | None = Field(description="확정된 장애 원인. 미확정이면 null")
    pending_checks: list[str]

print(incident_report)
print("운영 수칙:\n", runbook)


## 1. 인수인계 State와 초기 입력을 만듭니다

노드가 공유할 원문·수칙·추출 결과·컨텍스트·최종 노트를 정합니다.

**작성 방식:** 전체 코드 작성

- `IncidentState(TypedDict)`에 `report_text: str`, `runbook: str`, `facts: dict`, `briefing_context: str`, `handoff: str`를 선언하세요.
- `incident_input`의 원문과 수칙은 준비한 incident_report·runbook을 사용하세요.
- facts의 키는 symptoms(증상 목록), affected_scope(영향 범위), confirmed_cause(확정 원인), pending_checks(확인할 항목 목록)입니다.
- facts의 초기값은 symptoms·pending_checks가 빈 리스트, affected_scope가 빈 문자열, confirmed_cause가 None입니다. 나머지 결과 문자열도 비워 둡니다.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
# facts: 키는 symptoms(증상 목록), affected_scope(영향 범위),
# confirmed_cause(확정 원인 또는 None), pending_checks(확인할 항목 목록)
# 초기 State: 키는 필드 이름, 값은 원문 또는 각 결과의 초기값
# 여기에 코드를 작성하세요.


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
print("State 필드:", list(IncidentState.__annotations__))
print("초기 결과가 비어 있나요?", incident_input["briefing_context"] == "" and incident_input["handoff"] == "")
print("원문이 준비 입력과 같나요?", incident_input["report_text"] == incident_report)


## 2. LLM 사실 추출 노드를 만듭니다

LLM은 관찰 사실을 IncidentFacts 모델 객체로 반환합니다. State에는 model_dump()로 변환한 딕셔너리를 저장합니다. 배포 이력만으로 원인을 확정하면 안 됩니다.

**작성 방식:** 핵심 코드 채우기

- `facts_chain`은 제공한 facts_prompt와 `llm.with_structured_output(IncidentFacts, strict=True)`를 연결하세요.
- `extract_facts`에서 state의 report_text로 체인을 호출해 `facts`에 담으세요. `facts.model_dump()`로 딕셔너리로 변환해 State의 facts 필드만 반환하세요.
- 아직 함수를 호출하지 않습니다. 실행은 6번에서 그래프가 담당합니다.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

facts_prompt = ChatPromptTemplate.from_messages([
    ("system", "장애 보고서에서 관찰된 사실을 구조화하세요. "
     "symptoms에는 보고서에 관찰된 지연·오류와 수치를 담고, affected_scope에는 영향받은 서비스와 범위를 담으세요. "
     "confirmed_cause에는 확정된 원인만 쓰고, 원인이 미확정이면 null로 반환하세요. "
     "배포와 장애의 시간상 선후관계를 원인 확정으로 바꾸지 마세요. "
     "pending_checks에는 아직 확인되지 않은 원인·데이터 손실 여부·복구 예정 시각을 담으세요. "
     "완료된 복구 조치가 없다는 내용은 이미 확인된 상태이므로 완료 여부가 불명확한 것으로 바꾸지 마세요. "
     "보고서에 없는 담당자, 수치, 복구 시각, 조치 결과는 만들지 마세요."),
    ("human", "{report_text}"),
])
facts_chain = ...

def extract_facts(state: IncidentState):
    """보고서의 확인 사실과 미확인 항목을 facts로 반환합니다."""
    facts = ...
    return ...


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
print("출력 필드:", list(IncidentFacts.__annotations__))
print("노드 이름:", extract_facts.__name__)


## 3. 구조화 결과를 다음 노드에 전달합니다

다음 LLM이 읽도록 facts를 JSON 문자열로 바꿉니다. 이 변환에는 모델이 필요하지 않습니다.

**작성 방식:** 핵심 코드 채우기

- `briefing_context`는 `json.dumps`로 state의 facts를 문자열로 바꾼 값입니다. `ensure_ascii=False`, `indent=2`를 사용하세요.
- `format_briefing`은 briefing_context 한 필드만 반환하세요.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
def format_briefing(state: IncidentState):
    """앞 노드의 구조화 결과를 다음 LLM이 읽을 문자열로 바꿉니다."""
    briefing_context = ...
    return ...


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
preview_update = format_briefing(incident_input)
print("반환 필드:", list(preview_update))
print("원문은 바뀌지 않았나요?", incident_input["report_text"] == incident_report)
print(preview_update["briefing_context"])


## 4. 근거에 기반한 인수인계 노드를 만듭니다

원 보고서·정리한 사실·운영 수칙을 함께 사용해 다음 담당자가 읽을 노트를 만듭니다.

**작성 방식:** 핵심 코드 채우기

- `handoff_chain`은 handoff_prompt와 llm을 연결하세요.
- `draft_handoff`는 State의 report_text·briefing_context·runbook을 같은 이름으로 전달해 `response`를 받습니다.
- 반환값은 `{"handoff": response.text}`입니다. 기존 State를 직접 수정하지 마세요.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
handoff_prompt = ChatPromptTemplate.from_messages([
    ("system", "교대 담당자가 이어서 확인할 수 있도록 짧은 장애 인수인계 노트를 작성하세요. "
     "'확인된 사실', '아직 확인할 항목', '다음 확인'의 세 부분으로 나누세요. "
     "보고서와 구조화 결과의 사실만 쓰고, null인 원인은 '미확정'으로 표시하세요. "
     "장애 사실에는 [incident], 운영 수칙에 근거한 다음 확인에는 [runbook]을 표시하세요. "
     "다음 확인은 아직 수행하지 않은 계획입니다. 각 항목을 확인 필요·비교 필요·공유 필요처럼 쓰고 확인함·비교함·공유함으로 쓰지 마세요. "
     "최근 배포를 확정 원인으로 단정하지 마세요. 결제 데이터 손실 유무와 복구 시각은 아직 모릅니다. "
     "점검이나 복구를 이미 수행했다고 말하지 말고, 새로운 담당자·수치·시각을 만들지 마세요.\n"
     "운영 수칙:\n{runbook}"),
    ("human", "원 보고서:\n{report_text}\n\n구조화한 사실:\n{briefing_context}"),
])
handoff_chain = ...

def draft_handoff(state: IncidentState):
    """구조화한 사실과 운영 수칙으로 인수인계 노트를 작성합니다."""
    response = ...
    return ...


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
print("답변 노드:", draft_handoff.__name__)
print("프롬프트 입력:", handoff_prompt.input_variables)


## 5. 세 노드를 고정 엣지로 연결합니다

분류 경로를 고를 필요 없이 모든 보고서를 같은 순서로 처리합니다.

**작성 방식:** 핵심 코드 채우기

- `incident_builder`는 IncidentState를 사용하는 StateGraph입니다. 노드 등록 코드는 제공합니다.
- START -> extract_facts -> format_briefing -> draft_handoff -> END 순서로 연결하세요.
- `incident_graph`는 builder의 compile 결과입니다. 아래 이미지에서 노드 순서를 확인하세요.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
from langgraph.graph import END, START, StateGraph

incident_builder = ...
incident_builder.add_node("extract_facts", extract_facts)
incident_builder.add_node("format_briefing", format_briefing)
incident_builder.add_node("draft_handoff", draft_handoff)

# 노드 사이의 실행 흐름 연결
...
incident_graph = ...


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
display(Image(incident_graph.get_graph().draw_mermaid_png()))
print("노드:", list(incident_graph.get_graph().nodes))


## 6. 실행 결과를 원 보고서와 대조합니다

실제 GPT를 두 번 호출해 구조화 결과와 인수인계 노트를 받습니다.

**작성 방식:** 핵심 코드 채우기

- `incident_result`는 incident_graph에 incident_input을 invoke한 결과입니다.
- symptoms의 수치와 영향 범위를 원 보고서와 대조하세요. confirmed_cause는 None이어야 합니다.
- 노트가 데이터 손실이나 복구 완료를 단정하지 않고 [incident]·[runbook] 근거를 구분하는지 읽으세요.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
incident_result = ...
print("구조화한 사실:")
print(json.dumps(incident_result["facts"], ensure_ascii=False, indent=2))
print("인수인계 노트:")
display(Markdown(incident_result["handoff"]))


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
print("원인이 미확정으로 남았나요?", incident_result["facts"]["confirmed_cause"] is None)
print("관찰 증상이 있나요?", bool(incident_result["facts"]["symptoms"]))
print("확인할 항목:", incident_result["facts"]["pending_checks"])
print("초기 입력은 유지됐나요?", incident_input["handoff"] == "" and incident_input["facts"]["symptoms"] == [])
print("두 근거 표시가 있나요?", "[incident]" in incident_result["handoff"] and "[runbook]" in incident_result["handoff"])


## 7. 대화를 누적할 State를 만듭니다

방금 만든 노트를 근거로 담당자의 후속 질문을 받습니다. 답변마다 앞 대화가 유지되어야 합니다.

**작성 방식:** 전체 코드 작성

- `ReviewState(TypedDict)`에 report_text·runbook·handoff는 str로, messages는 `Annotated[list[AnyMessage], add_messages]`로 선언하세요.
- `review_input`에는 incident_result의 같은 이름 세 필드를 넣고 messages는 빈 리스트로 시작하세요.
- State와 초기 입력 전체를 직접 작성하세요.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

# 여기에 코드를 작성하세요.


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
print("앞 결과가 연결됐나요?", review_input["handoff"] == incident_result["handoff"])
print("초기 메시지:", review_input["messages"])
print("messages 선언:", ReviewState.__annotations__["messages"])


## 8. 질문을 입력받는 노드를 만듭니다

입력 노드가 질문 하나를 HumanMessage로 반환합니다. 종료 경로 함수는 제공합니다.

**작성 방식:** 핵심 코드 채우기

- `question`에는 `input("추가 질문 (/종료로 끝내기): ").strip()`의 결과를 담으세요.
- `read_question`은 `{"messages": [HumanMessage(content=question)]}`을 반환하세요.
- `/종료`도 기록에는 남지만 route_review가 END로 보내 GPT 답변은 만들지 않습니다.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
from langchain_core.messages import HumanMessage

def read_question(state: ReviewState):
    """입력한 질문 한 개를 대화 기록에 추가합니다."""
    question = ...
    print("사용자 입력:", question)
    return ...

# 종료 명령은 답변 노드로 보내지 않습니다.
def route_review(state: ReviewState):
    if state["messages"][-1].content == "/종료":
        return "종료"
    return "계속"


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
print("입력 노드:", read_question.__name__)
print("종료 입력의 경로:", route_review({**review_input, "messages": [HumanMessage(content="/종료")]}))
print("일반 질문의 경로:", route_review({**review_input, "messages": [HumanMessage(content="원인은 확인됐나요?")]}))


## 9. 대화 기록을 읽는 답변 노드를 만듭니다

마지막 질문의 지시어를 앞 대화와 연결하면서, 실제 사실은 원 보고서·운영 수칙으로 확인합니다.

**작성 방식:** 핵심 코드 채우기

- `review_chain`은 review_prompt와 llm을 연결하세요.
- `response`는 State의 report_text·runbook·handoff·messages를 같은 이름으로 전달한 결과입니다.
- 반환값은 새 답변 하나인 `{"messages": [response]}`입니다. 전체 기존 messages를 다시 붙이지 마세요.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
review_prompt = ChatPromptTemplate.from_messages([
    ("system", "장애 인수인계를 검토하는 담당자의 마지막 질문에 한국어로 짧게 답하세요. "
     "대화에서 앞서 언급한 대상이나 지시어는 앞 대화와 연결하세요. "
     "원 보고서와 운영 수칙에 없는 사실은 만들지 마세요. 확정되지 않은 원인·데이터 손실·복구 시각은 미확정이라고 답하세요. "
     "앞서 만든 인수인계 노트는 요약이며, 사실의 근거는 원 보고서와 운영 수칙입니다. "
     "사실에는 [incident], 운영 수칙에는 [runbook]을 표시하세요. "
     "점검·승인·복구를 실제 수행했다고 말하지 마세요.\n"
     "원 보고서:\n{report_text}\n운영 수칙:\n{runbook}\n인수인계 노트:\n{handoff}"),
    ("placeholder", "{messages}"),
])
review_chain = ...

def answer_review(state: ReviewState):
    """누적된 대화를 읽고 새 답변 메시지 한 개만 반환합니다."""
    response = ...
    print("담당자 답변:")
    display(Markdown(response.text))
    return ...


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
print("답변 노드:", answer_review.__name__)
print("대화 자리:", review_prompt.optional_variables)


## 10. 질문과 답변이 이어지는 그래프를 실행합니다

입력 -> 답변 -> 입력을 반복하며 이번 invoke 실행 동안 대화를 유지합니다.

**작성 방식:** 핵심 코드 채우기

- `review_builder`는 ReviewState를 사용하고 두 노드 등록은 제공 코드를 사용합니다.
- START -> read_question, answer_review -> read_question을 고정 엣지로 연결하세요.
- read_question 뒤에 route_review와 `{"계속": "answer_review", "종료": END}`를 add_conditional_edges로 등록하세요.
- `review_graph`를 compile하고 `review_result`에 review_input을 invoke한 결과를 담으세요. `config={"recursion_limit": 1000}`을 사용합니다.
- 순서대로 `시간 초과는 몇 건이고 원인이 확정됐나요?`, `그 원인을 확인하려면 다음으로 무엇을 봐야 하나요?`, `/종료`를 입력하세요.
- 두 번째 답변이 앞의 장애 원인 확인에 대한 질문으로 이어지는지 읽으세요.

**확인:** 아래 출력과 원 보고서를 비교하세요.


In [ ]:
review_builder = ...
review_builder.add_node("read_question", read_question)
review_builder.add_node("answer_review", answer_review)

# 사용자 입력 후 답변하거나 종료하고, 답변 뒤에는 다음 입력을 받습니다.
...
review_graph = ...
display(Image(review_graph.get_graph().draw_mermaid_png()))

review_result = ...


### 작성한 결과를 확인합니다

이 셀은 모델을 새로 호출하지 않습니다.


In [ ]:
print("메시지 수:", len(review_result["messages"]))
print("질문 2개·답변 2개·종료 입력으로 5개인가요?", len(review_result["messages"]) == 5)
print("메시지 역할:", [message.type for message in review_result["messages"]])
for message in review_result["messages"]:
    print(message.type, ":", message.text)
print("시작 입력의 기록은 비어 있나요?", review_input["messages"] == [])


## 마지막으로 설명해 봅니다

- State에 원문과 운영 수칙을 함께 둔 이유는 무엇인가요?
- 구조화 출력의 형식이 맞아도 확인해야 할 내용은 무엇인가요?
- format_briefing을 사실 추출보다 먼저 연결하면 어떤 데이터를 읽나요?
- 새 메시지 하나만 반환하는데 앞 대화가 남는 이유를 설명하세요.
